In [2]:
import numpy as np
import pandas as pd
import pickle
from statsmodels.tsa.api import VAR
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from statsmodels.tsa.base.datetools import dates_from_str

將檔案讀進來

In [3]:
file_number = 3 #看要多少個檔案，可以填入1-7

In [4]:
x_hat_dict_all = {}

for i in range(1, file_number+1):
    filename = f'result/x_hat{i}_dict.pkl'
    with open(filename, 'rb') as file:
        x_hat_dict_all[f'x_hat{i}_dict'] = pickle.load(file)

# for dict_key, inner_dict in x_hat_dict_all.items():
#     for key, value_dict in inner_dict.items():
#         for sub_key, array_value in value_dict.items():
#             x_hat_dict_all[dict_key][key][sub_key] = array_value.values.reshape(-1)

抓出每個檔案有多少col

In [5]:
x_hat_columns = {}

for dict_key, inner_dict in x_hat_dict_all.items():
    for key, value_dict in inner_dict.items():
        for sub_key, array_value in value_dict.items():     
            x_hat_columns[key] = len(array_value.columns)

整理資料找最佳權重

In [6]:
x_hat_add = {}

list_1 = x_hat_dict_all['x_hat1_dict']
for d in list_1:
    x_hat_add[d] = {}
    for i in range(file_number):
        var_i = i + 1
        x_hat_add[d][f'x_hat{var_i}_dict'] = {}
        flag = True
        for j in range(var_i):
            if flag:
                merged_df = pd.concat([x_hat_dict_all[f'x_hat{var_i}_dict'][d][f'{j}']], axis=0)
                flag = False
            else:
                merged_df = pd.concat([merged_df, x_hat_dict_all[f'x_hat{var_i}_dict'][d][f'{j}']], axis=0)
        x_hat_add[d][f'x_hat{var_i}_dict'] = merged_df.groupby(merged_df.index).mean()

In [7]:
x_hat_add['80057524n.csv.gz']['x_hat2_dict']

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,46.783053,66.170932,126.080746,0.753601,89.826972,1.825397,0.330655,63.625032,86.099206,392.000002,92.000002,484.999996,24.538912,1.362036,1.683570,90.828570,92.432782
1,48.495028,66.877480,123.580264,0.834107,89.041177,2.589356,0.538717,64.473254,76.105979,398.494119,91.458826,491.223525,24.483193,0.799350,1.272630,92.682286,91.050570
2,46.950650,66.087811,126.652989,9.614554,86.317188,1.088725,0.938118,63.616819,58.550132,411.090200,90.887537,504.661628,23.330532,0.702558,1.196936,78.998235,90.672600
3,49.611211,68.957065,123.678756,5.499998,88.365915,2.079262,0.453203,64.468350,49.907467,412.761903,90.202379,505.261906,23.217361,0.763770,1.228951,85.298647,90.027109
4,47.024787,68.359607,126.810566,5.499999,90.288037,2.192121,0.682761,63.616277,76.261362,407.999999,85.637432,500.499999,23.831348,0.766895,1.249706,84.006927,92.298463
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13716,47.017622,70.845240,126.864015,5.499999,90.242118,2.128615,0.734924,63.602613,76.261362,407.999999,88.310585,500.499999,23.831348,0.748190,1.267594,84.006927,92.298463
13717,49.424393,67.192177,123.740572,5.499998,88.365915,2.308490,0.453203,64.464728,49.907467,412.761903,90.202379,505.261906,21.732976,0.755259,1.249902,85.298647,89.280714
13718,47.022493,68.804603,126.869058,9.742502,86.908826,1.088725,0.938118,63.662587,58.457336,411.090200,90.887537,504.661628,20.945077,0.675922,1.214715,78.998235,90.672600
13719,49.550896,67.360880,123.672864,-3.773233,89.041177,2.589356,0.578900,64.475042,76.008548,398.494119,91.458826,491.223525,24.483193,0.806450,1.293354,92.966790,91.050570


In [8]:
x_hat_by_variable = {}

list_1 = x_hat_dict_all['x_hat1_dict']
for d in list_1:
    x_hat_by_variable[d] = {}
    col_num = x_hat_columns[d]
    for c in range(col_num):
        flag = True
        for i in range(file_number):
            var_i = i + 1
            if flag:
                merged_df = pd.concat([x_hat_add[d][f'x_hat{var_i}_dict'][c]], axis=1)
                flag = False
            else:
                merged_df = pd.concat([merged_df, x_hat_add[d][f'x_hat{var_i}_dict'][c]], axis=1)

            x_hat_by_variable[d][c] = merged_df


In [9]:
x_hat_by_variable['80057524n.csv.gz'][1]

,1,1,1
0,75.936711,66.170932,67.365925
1,76.141177,66.877480,65.957691
2,76.133597,66.087811,69.914633
3,76.170750,68.957065,67.892816
4,76.154479,68.359607,67.636444
...,...,...,...
13716,75.943452,70.845240,68.208115
13717,75.923215,67.192177,69.208158
13718,75.506439,68.804603,69.906605
13719,75.865307,67.360880,67.994589


VAR

In [10]:
x_hat_weight_ar = {}

for dict_key, inner_dict in x_hat_by_variable.items():
    x_hat_weight_ar[dict_key] = {}  
    for key, value_dict in inner_dict.items():
        # 定義一個範圍，例如 1 到 10 的階數
        orders_to_try = range(1, 11)

        # 儲存每個模型的 AIC 值
        aic_values = []

        # 嘗試不同的階數
        for order in orders_to_try:
            model = VAR(value_dict)
            results = model.fit(order)
            aic_values.append(results.aic)

        # 找到 AIC 值最小的階數
        best_order = orders_to_try[np.argmin(aic_values)]

        # 創建 VAR 模型
        model = VAR(value_dict)
        results = model.fit(best_order)

        # 提取每行的權重
        coefficients = results.coefs

        # 重新整形為 2D 陣列
        coefficients_2d = coefficients.T.reshape(-1, len(value_dict.columns))

        # 轉換為 DataFrame
        coefficients_df = pd.DataFrame(coefficients_2d, columns=value_dict.columns)


        average_coefficients_df = coefficients_df.mean(axis=0)

        normalized_coefficients_df = average_coefficients_df / average_coefficients_df.sum()

        x_hat_weight_ar[dict_key][key] = normalized_coefficients_df.values


PCA

In [11]:
x_hat_weight_pca = {}

for dict_key, inner_dict in x_hat_by_variable.items():
    x_hat_weight_pca[dict_key] = {}  
    for key, value_dict in inner_dict.items():

        pca = PCA(n_components=None, svd_solver='full')

        # 将DataFrame转换为数组
        X = value_dict.values

        # 拟合PCA模型并转换数据
        X_pca = pca.fit_transform(X)

        # 获取每个主成分的权重
        weights = pca.components_

        # 对权重进行归一化处理，确保每个主成分的权重总和为1
        normalized_weights = normalize(weights, axis=1, norm='l1')[0]

        # 确保每个主成分的权重总和为1
        total_weight = np.sum(normalized_weights)
        normalized_weights /= total_weight

        x_hat_weight_pca[dict_key][key] = normalized_weights

將權重與原始檔案合併

In [29]:
weight_dict_type = 'AR' # 這裡看是要用VAR還是PCA

In [30]:
x_hat_dot = {}

# 選擇要使用的權重字典
use_weights = x_hat_weight_pca if weight_dict_type == 'PCA' else x_hat_weight_ar

for dict_key, inner_dict in x_hat_by_variable.items():
    x_hat_dot[dict_key] = {}
    for key, value_dict in inner_dict.items():
        x_hat_dot[dict_key][key] = np.dot(value_dict, use_weights[dict_key][key]) 

In [31]:
x_hat_reshape = {}

for dict_key, inner_dict in x_hat_by_variable.items():
    col_num = x_hat_columns[dict_key]
    flag = True
    for c in range(col_num):
        if flag:
                merged_df = pd.concat([pd.DataFrame(x_hat_dot[dict_key][c])], axis=1)
                flag = False
        else:
                merged_df = pd.concat([merged_df, pd.DataFrame(x_hat_dot[dict_key][c])], axis=1)
        merged_df.columns = range(len(merged_df.columns))
        x_hat_reshape[dict_key] = merged_df

In [32]:
x_hat_reshape_line = {}

for key, val in x_hat_reshape.items():
    x_hat_reshape_line[key] = val.values.reshape(-1)

In [33]:
# reshape_x_hat = {}

# list_1 = x_hat_dict_all['x_hat1_dict']
# for d in list_1:
#     col_num = x_hat_columns[d]
#     reshape_x_hat[d] = {}
#     for c in range(col_num):
#         flag = True
#         for i in range(2):
#             var_i = i + 1
#             for j in range(var_i):
#                 if flag:
#                     merged_df = pd.concat([x_hat_dict_all[f'x_hat{var_i}_dict'][d][f'{j}'][c]], axis=1)
#                     flag = False
#                 else:
#                     merged_df = pd.concat([merged_df, x_hat_dict_all[f'x_hat{var_i}_dict'][d][f'{j}'][c]], axis=1)
#         reshape_x_hat[d][c] = merged_df
#                 # print(var_i, "======", d, "======", j)

抓出boolen為true的來進行驗證

In [34]:
with open('result/eval_value_dict.pkl', 'rb') as file:
    eval_value_dict = pickle.load(file)
for key, val in eval_value_dict.items():
    eval_value_dict[key] = val.reshape(-1)

將原始的x_hat拉成直線並抓出10%驗證

In [35]:
x_hat_original_line = {}

for key, val in x_hat_dict_all['x_hat1_dict'].items():
    x_hat_original_line[key] = val['0'].values.reshape(-1)

In [36]:
x_hat_original = {}

for key, val in x_hat_original_line.items():
    x_hat_original[key] = val[eval_value_dict[key]]

將做出來的x_hat拉成直線並抓出10%驗證

In [37]:
x_hat_reshape_line = {}

for key, val in x_hat_reshape.items():
    x_hat_reshape_line[key] = val.values.reshape(-1)

In [38]:
x_hat_result = {}

for key, val in x_hat_reshape_line.items():
    x_hat_result[key] = val[eval_value_dict[key]]

將原始沒有補過的檔案拉成直線並抓10%驗證

In [39]:
with open('result/data_frames.pkl', 'rb') as file:
    data_frames = pickle.load(file)

# 把最後一行刪掉
for key in data_frames:
    data_frames[key] = data_frames[key].iloc[:-1]

data_frames = {key: value for key, value in data_frames.items() if len(value) >= 100}

In [40]:
data_frames_line = {}

for key, val in data_frames.items():
    data_frames_line[key] = val.values.reshape(-1)

In [41]:
original_df = {}

for key, val in data_frames_line.items():
    original_df[key] = val[eval_value_dict[key]]

In [42]:
# mse

def calculate_mse(imputed, actual):
    if actual.shape == imputed.shape:
        mse = np.mean((imputed - actual) ** 2)
        return mse
    else:
        raise ValueError(f'Input shapes do not match: {actual.shape} and {imputed.shape}.')
    

# mae

def calculate_mae(imputed, actual):
    if actual.shape == imputed.shape:
        mae = np.mean(np.abs(imputed - actual))
        return mae
    else:
        raise ValueError(f'Input shapes do not match: {actual.shape} and {imputed.shape}.')

實驗結果

In [43]:
x_hat_result_mse = {}
for key, val in original_df.items():
    x_hat_result_mse[key] = calculate_mse(x_hat_result[key], original_df[key])

print("mse:", sum(x_hat_result_mse.values()) / len(x_hat_result_mse))

x_hat_result_mae = {}
for key, val in original_df.items():
    x_hat_result_mae[key] = calculate_mae(x_hat_result[key], original_df[key])

print("mae:", sum(x_hat_result_mae.values()) / len(x_hat_result_mae))

mse: 122.15946021921606
mae: 5.278479592038886


原始

In [44]:
x_hat_original_mse = {}
for key, val in original_df.items():
    x_hat_original_mse[key] = calculate_mse(x_hat_original[key], original_df[key])

print("mse:", sum(x_hat_original_mse.values()) / len(x_hat_original_mse))

x_hat_original_mae = {}
for key, val in original_df.items():
    x_hat_original_mae[key] = calculate_mae(x_hat_original[key], original_df[key])

print("mae:", sum(x_hat_original_mae.values()) / len(x_hat_original_mae))

mse: 124.79634348504116
mae: 5.308948307970866


In [28]:
len(x_hat_dict_all['x_hat1_dict'])

175